# CDC Pipeline — Sample Analytics Queries

This notebook runs sample queries against the ClickHouse warehouse that the streaming Spark job populates.

**What to look for**
1. The `analytics.*` raw tables grow with every CDC event (insert, update, *and* delete — deletes are kept with `_is_deleted=1`).
2. The `analytics.v_*` views always return the latest-state of each row, hiding the CDC plumbing.
3. The `analytics.cdc_dead_letter` table stays empty unless the upstream emits something the Spark job can't parse.

If you re-run cells over time you'll see counts grow — that's the data generator producing more activity.

In [ ]:
import os
from clickhouse_driver import Client
import pandas as pd

client = Client(
    host=os.environ.get('CLICKHOUSE_HOST', 'clickhouse'),
    port=int(os.environ.get('CLICKHOUSE_NATIVE_PORT', '9000')),
    user=os.environ.get('CLICKHOUSE_USER', 'default'),
    password=os.environ.get('CLICKHOUSE_PASSWORD', 'clickpass'),
    database=os.environ.get('CLICKHOUSE_DB', 'analytics'),
)

def q(sql):
    rows, cols_with_types = client.execute(sql, with_column_types=True)
    cols = [c[0] for c in cols_with_types]
    return pd.DataFrame(rows, columns=cols)

print('Connected to ClickHouse, current database:', client.execute('SELECT currentDatabase()')[0][0])

## 1. Row counts per table (raw + latest-state)

Raw tables grow with every CDC event. The views show *current* state only.

In [ ]:
q('''
    SELECT 'customers'   AS table, count() AS raw_rows FROM customers
    UNION ALL SELECT 'orders',      count() FROM orders
    UNION ALL SELECT 'order_items', count() FROM order_items
    UNION ALL SELECT 'dead_letter', count() FROM cdc_dead_letter
''')

In [ ]:
q('''
    SELECT 'v_customers'   AS view, count() AS rows FROM v_customers
    UNION ALL SELECT 'v_orders',      count() FROM v_orders
    UNION ALL SELECT 'v_order_items', count() FROM v_order_items
''')

## 2. Orders per minute (last 30 minutes)

Demonstrates time-series aggregation. Watch this re-run as the generator produces more events.

In [ ]:
q('''
    SELECT toStartOfMinute(created_at) AS minute,
           count()                     AS orders,
           sum(total_cents)/100        AS revenue_eur
    FROM v_orders
    WHERE created_at >= now() - INTERVAL 30 MINUTE
    GROUP BY minute
    ORDER BY minute DESC
''')

## 3. Order status funnel

Where are orders sitting in the lifecycle?

In [ ]:
q('''
    SELECT status, count() AS n, round(100 * n / sum(n) OVER (), 1) AS pct
    FROM v_orders
    GROUP BY status
    ORDER BY n DESC
''')

## 4. Top 5 customers by lifetime spend

In [ ]:
q('''
    SELECT c.id, c.full_name, c.country,
           count(o.id)              AS orders,
           sum(o.total_cents) / 100 AS lifetime_spend_eur
    FROM v_orders o
    JOIN v_customers c ON c.id = o.customer_id
    WHERE o.status IN ('paid','shipped','delivered')
    GROUP BY c.id, c.full_name, c.country
    ORDER BY lifetime_spend_eur DESC
    LIMIT 5
''')

## 5. Best-selling SKUs

In [ ]:
q('''
    SELECT sku,
           sum(qty)                                  AS units_sold,
           sum(qty * unit_price_cents) / 100         AS gross_eur
    FROM v_order_items
    GROUP BY sku
    ORDER BY units_sold DESC
    LIMIT 10
''')

## 6. CDC operations breakdown

What's flowing through the pipeline right now? `c`=create, `u`=update, `d`=delete, `r`=snapshot read.

In [ ]:
q('''
    SELECT 'orders' AS t, _op, count() AS n FROM orders GROUP BY _op
    UNION ALL
    SELECT 'customers', _op, count() FROM customers GROUP BY _op
    UNION ALL
    SELECT 'order_items', _op, count() FROM order_items GROUP BY _op
    ORDER BY t, _op
''')

## 7. Pipeline latency (Kafka offset → ingest)

Difference between the source event timestamp and the moment Spark inserted the row into ClickHouse. Tells you how live your warehouse really is.

In [ ]:
q('''
    SELECT round(quantile(0.5)(latency_ms))  AS p50_ms,
           round(quantile(0.95)(latency_ms)) AS p95_ms,
           round(quantile(0.99)(latency_ms)) AS p99_ms,
           round(max(latency_ms))            AS max_ms
    FROM (
        SELECT toUnixTimestamp64Milli(_ingested_at) - _op_ts_ms AS latency_ms
        FROM orders
        WHERE _ingested_at >= now() - INTERVAL 10 MINUTE
    )
''')